# NutriVision EDA v2 (알레르기 + 나트륨·당)

**순서대로 위에서부터 실행.** 한 파일 안에 필요한 코드가 모두 들어 있다(별도 .py 파일 없음).

| 파트 | 하는 일 | 시간 |
|---|---|---|
| 준비 | 드라이브 연결, food-101.tar.gz 압축 해제 | 5~10분 |
| A | 사진 데이터 확인 (장수, 크기, 헷갈리는 쌍) | 1~2분 |
| B | 레시피 CSV 컬럼부터 확인, 불러오기 | 1~2분 |
| C | 음식 이름으로 레시피 연결 (규칙으로 정리) | 3~5분 |
| D | 음식별 알레르기 비율 | 1분 |
| E | 음식별 나트륨·당 (영양 컬럼이 있을 때만) | 1분 |

분석은 늘 **확인 → 발견 → 결정** 순서. 결과 그림과 표는 드라이브의 `eda_out` 폴더에 저장된다.

※ 알레르기·매칭 규칙은 간단 버전이라 팀원 노트북 결과와 숫자가 조금 다를 수 있다.

In [ ]:
import os, re, glob, ast, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import drive
drive.mount('/content/drive')

# 한글 그림 글자용 폰트
try:
    import subprocess
    subprocess.run('apt-get -qq -y install fonts-nanum', shell=True, check=True, capture_output=True)
    from matplotlib import font_manager as fm
    fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    plt.rcParams['font.family'] = 'NanumGothic'
except Exception as e:
    print('폰트 설치 실패(그림 글자가 깨질 수 있음):', e)
plt.rcParams.update({'axes.unicode_minus': False, 'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

def find(name):
    for pat in ['*', '*/*', '*/*/*']:
        hit = glob.glob(f'/content/drive/MyDrive/{pat}/{name}')
        if hit:
            return hit[0]
    raise FileNotFoundError(name)

TAR, CSV = find('food-101.tar.gz'), find('1M_Final.csv')
print(TAR)
print(CSV)
OUT = os.path.join(os.path.dirname(TAR), 'eda_out')   # 결과는 드라이브 eda_out 폴더에 저장
os.makedirs(OUT, exist_ok=True)

def save(name):
    plt.savefig(os.path.join(OUT, name + '.png'), bbox_inches='tight')
    plt.show()


In [ ]:
# 5GB 압축 해제: 5~10분. 이미 풀려 있으면 건너뜀
if not os.path.exists('/content/food-101/images'):
    !tar -xzf "{TAR}" -C /content
ROOT = '/content/food-101'
classes = open(f'{ROOT}/meta/classes.txt').read().split()
print(len(classes), classes[:5])


## A. 사진 데이터 (Food-101)
확인할 것: ① 장수·균형 ② 크기·비율 ③ 헷갈리는 음식

In [ ]:
# ① 장수·균형
counts = pd.Series({c: len(os.listdir(f'{ROOT}/images/{c}')) for c in classes})
print('사진 수:', counts.sum(), '| 클래스당', counts.min(), '~', counts.max())
split = {s: sum(1 for _ in open(f'{ROOT}/meta/{s}.txt')) for s in ['train', 'test']}
print('공식 분할:', split)
plt.figure(figsize=(12, 3.2)); plt.bar(range(len(counts)), counts.values)
plt.ylim(0, 1200); plt.title('클래스별 사진 수 (Food-101)'); plt.xlabel('101개 클래스'); save('A1_class_counts')

# ② 크기·비율·색상 모드: 5,000장 무작위 표본
random.seed(0)
files = [(c, f) for c in classes for f in os.listdir(f'{ROOT}/images/{c}')]
rows = []
for c, f in random.sample(files, 5000):
    with Image.open(f'{ROOT}/images/{c}/{f}') as im:
        rows.append((im.width, im.height, im.mode))
sz = pd.DataFrame(rows, columns=['w', 'h', 'mode'])
sz['short'], sz['long'], sz['aspect'] = sz[['w', 'h']].min(axis=1), sz[['w', 'h']].max(axis=1), sz.w / sz.h
print('긴 변 512px 비율:', (sz.long == 512).mean(), '| 짧은 변 224 미만:', (sz.short < 224).sum(), '장 / 5000')
print('정사각형 비율:', (sz.aspect.round(2) == 1).mean(), '| 색상 모드:', sz['mode'].value_counts().to_dict())
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].hist(sz.short, bins=30); ax[0].axvline(224, ls='--', c='k'); ax[0].set_title('짧은 변 길이 (점선 = 224px)')
ax[1].hist(sz.aspect.clip(0.4, 2), bins=30); ax[1].set_title('가로/세로 비율'); save('A2_size_aspect')


In [ ]:
# ③ 헷갈리는 음식 쌍
def pics(c, n):
    return [Image.open(f'{ROOT}/images/{c}/{f}').convert('RGB').resize((200, 200)) for f in random.sample(os.listdir(f'{ROOT}/images/{c}'), n)]

PAIRS = [('ramen', 'pho'), ('gyoza', 'dumplings'), ('steak', 'filet_mignon'), ('chocolate_cake', 'chocolate_mousse')]
fig, ax = plt.subplots(len(PAIRS), 6, figsize=(12, 8))
for i, (a, b) in enumerate(PAIRS):
    for j, im in enumerate(pics(a, 3) + pics(b, 3)):
        ax[i][j].imshow(im); ax[i][j].axis('off')
        if j in (0, 3):
            ax[i][j].set_title(a if j == 0 else b, loc='left')
plt.suptitle('헷갈리는 음식 쌍 (왼쪽 3장 vs 오른쪽 3장)'); save('A3_confusing_pairs')


## B. 레시피 CSV (1M_Final)
먼저 **컬럼이 무엇인지**부터 본다. 영양(나트륨·당) 컬럼이 있는지가 이 노트북의 핵심 확인 사항.

In [ ]:
head = pd.read_csv(CSV, nrows=5)
print(head.shape[1], '개 컬럼:', head.columns.tolist())
head.T


In [ ]:
cols = head.columns.tolist()
pick = lambda pat: [c for c in cols if re.search(pat, c, re.I)]
TITLE_COL = (pick('^title$') or pick('title|name') or [None])[0]
ING_COL = (pick('^ner$') or pick('ner') or pick('ingredient') or [None])[0]
NUTR = {'sodium': pick('sodium|natrium'), 'sugar': pick('sugar'), 'calories': pick('calor|energy|kcal')}
print('제목:', TITLE_COL, '| 재료:', ING_COL)
print('영양 후보:', NUTR)
# 자동 선택이 틀렸으면 아래처럼 직접 지정하고 이 셀부터 다시 실행
# TITLE_COL = 'title'; ING_COL = 'NER'; NUTR = {'sodium': ['sodium_mg'], 'sugar': ['sugars_g'], 'calories': ['calories']}
NUTR = {k: v[0] for k, v in NUTR.items() if v}

use = [TITLE_COL, ING_COL] + list(NUTR.values())
df = pd.read_csv(CSV, usecols=use, encoding='utf-8', encoding_errors='replace')
print(df.shape)
print(df.isna().mean().round(3))
df[list(NUTR.values())].describe().T if NUTR else print('영양 컬럼을 찾지 못함 → 나트륨·당 분석은 건너뜀')


## C. 음식 이름으로 레시피 연결
제목에 `ramen`이 들어간 레시피를 찾되, `Ramen Noodle Salad`처럼 **이름만 같은 다른 요리**는 규칙으로 뺀다. (3~5분)

In [ ]:
ALIAS = {'gyoza': ['gyoza', 'potsticker', 'pot sticker'], 'cup_cakes': ['cupcake'], 'hamburger': ['hamburger'],
         'french_fries': ['french fries'], 'spring_rolls': ['spring roll'], 'beef_tartare': ['beef tartare', 'steak tartare'],
         'eggs_benedict': ['eggs benedict', 'egg benedict'], 'macarons': ['macaron']}
FORMS = ['soup', 'salad', 'casserole', 'dip', 'sauce', 'stew', 'pie', 'sandwich', 'wrap', 'pizza', 'slaw', 'dressing', 'chili', 'cake', 'bread', 'pasta', 'burger']

def rx(words):
    return '(?<![a-z])(?:' + '|'.join(re.escape(w.rstrip('s')) + 's?' for w in words) + ')(?![a-z])'

head_t = df[TITLE_COL].fillna('').str.lower().str.split(' with ').str[0]   # 'with' 앞부분이 진짜 요리 이름
raw, final = {}, {}
for c in classes:
    words = ALIAS.get(c, [c.replace('_', ' ')])
    toks = {t.rstrip('s') for w in words for t in w.split()}
    bad = [f for f in FORMS if f not in toks]          # 이 음식 이름에 없는 '다른 요리 형태' 단어
    r = head_t.str.contains(rx(words), regex=True)
    raw[c] = r
    final[c] = r & ~head_t.str.contains(rx(bad), regex=True)
    if c == 'dumplings':
        final[c] &= ~head_t.str.contains('and dumpling', regex=False)   # 미국식 치킨 앤 덤플링 제외

cnt = pd.DataFrame({'raw': {c: int(raw[c].sum()) for c in classes}, 'final': {c: int(final[c].sum()) for c in classes}})
print('이름 일치', cnt.raw.sum(), '→ 규칙 적용 후', cnt.final.sum(), '(제외', cnt.raw.sum() - cnt.final.sum(), ')')
print('레시피 수 최소/중앙/최대:', cnt.final.min(), cnt.final.median(), cnt.final.max())
print('30개 미만(정보 부족):', cnt.index[cnt.final < 30].tolist())
for c in ['dumplings', 'ramen', 'tacos']:
    print(c, '제외 예:', df.loc[raw[c] & ~final[c], TITLE_COL].head(3).tolist())
cnt.to_csv(os.path.join(OUT, 'class_recipe_counts.csv'))

s = cnt.final.sort_values(ascending=False)
plt.figure(figsize=(12, 3.2)); plt.bar(range(len(s)), s.values, color='#2a78d6'); plt.yscale('log')
plt.axhline(30, ls='--', c='k'); plt.text(60, 36, '30개 미만 = 정보 부족')
plt.title('음식별 매칭 레시피 수 (로그)'); save('B1_recipe_counts')


## D. 음식별 알레르기 비율
재료 이름을 한국 알레르기 표시 19항목에 연결하고, 음식마다 **그 재료가 든 레시피의 비율**을 센다.

In [ ]:
pairs = pd.concat([pd.DataFrame({'idx': final[c][final[c]].index, 'label': c}) for c in classes], ignore_index=True)

def parse(x):
    try:
        return ' , '.join(ast.literal_eval(x))
    except Exception:
        return str(x)

ing = df.loc[pairs.idx.unique(), ING_COL].map(parse).str.lower()   # 매칭된 레시피만 재료 문자열로 변환

RULES = {   # 한국 알레르기 표시 19항목: (포함 단어, 제외 단어). 간단 규칙이라 팀 규칙과 다를 수 있음
    '알류': ('(?<![a-z])eggs?(?![a-z])|mayonnaise|meringue', ''),
    '우유': ('milk|cream|butter|cheese|yogurt|yoghurt|whey|ghee|custard', 'coconut milk|almond milk|soy milk|oat milk|rice milk|coconut cream|cream of tartar|peanut butter|almond butter|cocoa butter|nut butter'),
    '메밀': ('buckwheat|soba', ''),
    '땅콩': ('peanut', ''),
    '대두': ('soy|tofu|edamame|miso|tempeh', ''),
    '밀': ('wheat|flour|bread|pasta|spaghetti|macaroni|couscous|noodle|cracker|panko|semolina|bulgur|soy sauce', 'rice flour|rice noodle|corn flour|cornflour|almond flour|coconut flour|chickpea flour|buckwheat flour|gluten-free|gluten free'),
    '고등어': ('mackerel', ''),
    '게': ('(?<![a-z])crab(?![a-z])|crabmeat|crab meat', ''),
    '새우': ('shrimp|prawn', ''),
    '돼지고기': ('(?<![a-z])(?:pork|bacon|ham)(?![a-z])|sausage|pepperoni|prosciutto|chorizo|pancetta|salami|lard', ''),
    '복숭아': ('peach', ''),
    '토마토': ('tomato|ketchup|salsa|marinara', ''),
    '아황산류': ('sulfite|sulphite|dried apricot|white wine|red wine', ''),
    '호두': ('walnut', ''),
    '닭고기': ('chicken', ''),
    '쇠고기': ('beef|steak|veal|brisket|sirloin|ribeye|short rib', ''),
    '오징어': ('squid|calamari', ''),
    '조개류': ('clam|oyster|mussel|scallop|abalone|shellfish|cockle', ''),
    '잣': ('pine nut|pinenut|pignoli', ''),
}

def flag(text, inc, exc):
    t = text.str.replace(exc, ' ', regex=True) if exc else text
    return t.str.contains(inc, regex=True)

F = pd.DataFrame({k: flag(ing, *v) for k, v in RULES.items()})   # 레시피 × 19항목 (있다/없다)
print('항목별 등장 비율(매칭 레시피 전체):')
print(F.mean().round(3).sort_values(ascending=False).to_string())

ratio = F.loc[pairs.idx.values].set_index(pairs.label.values).groupby(level=0).mean().reindex(classes)
ratio.assign(n_recipes=cnt.final).to_csv(os.path.join(OUT, 'class_allergen_ratio.csv'), float_format='%.4f')

TRAVEL = ['pad_thai', 'pho', 'ramen', 'gyoza', 'dumplings', 'spring_rolls', 'sushi', 'takoyaki', 'fried_rice', 'hot_and_sour_soup',
          'peking_duck', 'samosa', 'falafel', 'paella', 'tacos', 'ceviche', 'fish_and_chips', 'clam_chowder', 'croque_madame', 'baklava']
R = ratio.loc[TRAVEL]
fig, ax = plt.subplots(figsize=(12, 7))
im = ax.imshow(R.values, cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(R.shape[1]), R.columns, rotation=45, ha='right')
ax.set_yticks(range(len(R)), [f'{c} ({cnt.final[c]})' for c in R.index])
for i in range(R.shape[0]):
    for j in range(R.shape[1]):
        if R.values[i, j] >= 0.2:
            ax.text(j, i, f'{R.values[i, j] * 100:.0f}', ha='center', va='center', fontsize=8, color='w' if R.values[i, j] > 0.55 else 'k')
fig.colorbar(im); ax.set_title('여행 음식 20종 × 알레르기 19항목: 그 재료가 든 레시피의 비율(%)')
save('C1_allergen_heatmap')


## E. 음식별 나트륨·당 (고혈압·당뇨용)
**꼭 확인:** 단위(mg/g)와, 숫자가 '1회분'인지 '레시피 전체'인지. 위 describe 표의 값 크기로 짐작할 수 있다.

In [ ]:
if not NUTR:
    print('영양 컬럼이 없어서 건너뜀 → 알레르기 중심으로 발표하고, 영양은 향후 계획으로')
else:
    N = df.loc[pairs.idx.values, list(NUTR.values())].apply(pd.to_numeric, errors='coerce')
    N.index = pairs.label.values
    med = N.groupby(level=0).median().reindex(classes).rename(columns={v: k for k, v in NUTR.items()})
    med.assign(n_recipes=cnt.final).to_csv(os.path.join(OUT, 'class_nutrition_median.csv'), float_format='%.2f')
    print(N.describe().T.round(1))   # ★ 단위와 1회분 여부 확인
    REF = {'sodium': 2000, 'sugar': 50}   # 하루 권고 기준(mg, g). 발표 전에 근거 확인하고 수정
    keys = [k for k in REF if k in med.columns]
    fig, axes = plt.subplots(1, max(len(keys), 1), figsize=(6 * max(len(keys), 1), 6), squeeze=False)
    for ax, k in zip(axes[0], keys):
        v = med.loc[TRAVEL, k].sort_values()
        ax.barh(v.index, v.values, color='#2a78d6'); ax.axvline(REF[k], ls='--', c='r')
        ax.set_title(f'{k} 중앙값 (빨간선 = 하루 권고 {REF[k]})')
    save('D1_nutrition_by_class')
    for k in keys:
        print(k, '하루 권고 대비 중앙값(%):', (med.loc[TRAVEL, k] / REF[k] * 100).round(0).to_dict())


## 서비스 미리보기
사진으로 음식 이름을 알았다고 가정하고, 내 알레르기와 비교한 결과가 이런 모양으로 나온다.

In [ ]:
# 임시 기준: 60% 이상 주의 / 20% 이상 확인 필요 / 레시피 30개 미만 정보 부족
def level(c, allergen):
    if cnt.final[c] < 30:
        return '정보 부족'
    v = ratio.loc[c, allergen]
    return '주의' if v >= 0.6 else '확인 필요' if v >= 0.2 else '해당 없음'

MY = ['땅콩', '우유']   # 예: 땅콩·우유 알레르기
for c in ['pad_thai', 'ramen', 'pho', 'baklava', 'takoyaki']:
    print(c, '(레시피', cnt.final[c], '개):', {a: level(c, a) for a in MY})
print('결과 파일 저장 위치:', OUT)
